## CRM Product Silver Ttransformation

### 00. Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DateType
from pyspark.sql.functions import col, trim
from pyspark.sql.window import Window

### 01. Build today's folder name and read from bronze table

In [0]:
today = datetime.today().strftime("%Y_%m_%d")   # e.g. 2026_06_23
bronze_table = f"`abc_business-core`.bronze_{today}.product_info"
df = spark.table(bronze_table)

### 02. Trim all string fields

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, F.trim(F.col(field.name)))

### 03. Filter invalid

In [0]:
df = df.filter(F.col("prd_id").isNotNull())

### 04. Product Key Parsing

In [0]:
df = df.withColumn("cat_id", F.regexp_replace(F.substring(col("prd_key"), 1, 5), "-", "_"))
df = df.withColumn("prd_key", F.substring(col("prd_key"), 7, F.length(col("prd_key"))))

### 05. Cost Cleanup

In [0]:
df = df.withColumn("prd_cost", F.coalesce(col("prd_cost"), F.lit(0)))

### 06. Product Line Normalization

In [0]:
df = (
    df
    # Normalize product line
    .withColumn(
        "prd_line",
        F.when(F.upper(col("prd_line")) == "M", "Mountain")
         .when(F.upper(col("prd_line")) == "R", "Road")
         .when(F.upper(col("prd_line")) == "S", "Other Sales")
         .when(F.upper(col("prd_line")) == "T", "Touring")
         .otherwise("n/a")
    )
)

### 07. Date Casting

In [0]:
df = df.withColumn("prd_start_dt", col("prd_start_dt").cast(DateType()))

### 08. Rename columns

In [0]:
RENAME_MAP = {
    "prd_id": "product_id",
    "cat_id": "category_id",
    "prd_key": "product_number",
    "prd_nm": "product_name",
    "prd_cost": "product_cost",
    "prd_line": "product_line",
    "prd_start_dt": "start_date",
    "prd_end_dt": "end_date"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

### 09. Dataframe Preview - Sanity check

In [0]:
df.limit(5).display()

### 06. Write Silver table

In [0]:
table_name = f"`abc_business-core`.silver.crm_products_{today}"
df.write.mode("overwrite").format("delta").saveAsTable(table_name)

### 07. Sanity checks of silver table

In [0]:
df = spark.sql(f"SELECT * FROM {table_name} LIMIT 5")
df.show()